# 手順5：認識の記載の再抽出（窓の終点＝index time）
- 対象：transport_final.csv の1,290例（main_analysis 列で後から絞る）
- 終点：index time
  - 搬送あり：搬送日時（時刻なしは日単位）
  - 搬送なし：死亡時刻＝当院の死亡確認カルテの記録時刻（なしは日単位）
  - 時刻あり：記録時刻 < index time の記録のみ。日単位：index 日当日までの記録を含め `same_day=1`
- 記録：全 visit_type（`source` 列＝医師訪問／看護／文書／その他）
- 抽出
  1. ARIS（ARIS・RIS・全角・アリスケア。「アリス」単独は除外。前後15字の「予定」は `yotei=1`）：全期間
  2. 暫定種語 A/B/C（予後の明言／家族への説明／看取り体制）：index 前 `SEED_WIN` 日以内
  3. 「今後のこと／今後について」欄の文言変化（定期訪問のみ）：index 前 `KONGO_WIN` 日以内
  4. ACP 系の語（DNAR・延命・蘇生・挿管・搬送しない・突然死・心肺停止・人生会議）を含む新しい文：index 前 `ACP_WIN` 日以内
- 判定（24h等）は手順6。ここでは目視用CSVまで。DBは読み取りのみ

In [ ]:
# 0. 設定
import sqlite3, re, unicodedata
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

DB      = "/Users/muna/Hana_research/data/db/Hana_Research.db"
POP_CSV = "/Users/muna/Hana_research/data/output/transport_20261023/transport_final.csv"
OUT     = Path(f"/Users/muna/Hana_research/data/output/recog_{date.today():%Y%m%d}")
OUT.mkdir(parents=True, exist_ok=True)

SEED_WIN  = 30   # 種語：index 前この日数以内（index 日を0日とする）
KONGO_WIN = 30   # 今後欄の変化：同上。None で全期間
YOTEI_WIN = 15   # ARIS の前後何文字以内の「予定」を yotei=1 とするか
CTX       = 60   # 目視用の前後文字数

DOC_VISIT_TYPES   = ["定期訪問", "緊急往診", "往診", "休日往診", "夜間往診", "深夜往診", "初診"]
NURSE_TYPES       = ["訪問看護", "訪問看護(介護)", "訪問看護(緊急)"]
DEATH_VISIT_TYPES = ["定期訪問", "緊急往診", "往診", "休日往診", "夜間往診", "深夜往診"]
DEATH_WORDS       = ["死亡確認", "死亡診断", "死亡を確認", "永眠"]
KONGO_VISIT_TYPE  = "定期訪問"

# NFKC 正規化後に照合
ARIS_RE = re.compile(r"(?<![A-Za-z])A?RIS(?=care|[^A-Za-z]|$)|アリス\s*・?\s*ケア", re.I)

SEED = {
    "A_prognosis": {
        "余命": r"余命",
        "予後": r"予後",
        "時間〜月単位": r"(?:時間|日|週|月)単位",
        "厳しい": r"厳しい|厳しく",
        "長くない": r"長くない|長くは(?:ない|もたない)",
        "終末期": r"終末期|ターミナル|エンドオブライフ|EOL",
        "最期": r"最期|死期",
        "看取り期": r"看取り期|臨死期",
    },
    "B_family_explanation": {
        "家族へ説明": r"(?:家族|妻|夫|息子|娘|長男|長女|次男|次女|嫁|KP|キーパーソン).{0,20}(?:説明|IC|ムンテラ|お話|話し)",
        "覚悟": r"覚悟",
        "急変の可能性": r"急変(?:の可能性|もあり|もあ|しうる|し得る|のリスク)",
        "亡くなる": r"亡くな|死亡の可能性|死が近",
    },
    "C_eol_plan": {
        "看取り": r"看取り|看取る|お看取り",
        "死亡時の連絡": r"(?:死亡|呼吸停止|心停止).{0,10}(?:時|際).{0,10}(?:連絡|対応)",
        "エンゼルケア": r"エンゼル",
    },
}
SEED_RE = [(cat, term, re.compile(p, re.I)) for cat, d in SEED.items() for term, p in d.items()]
NEG_RE  = re.compile(r"なし|無し|ない|せず|しない|否定|希望されず")   # 目安のみ。判定に使わない

# 今後欄：見出し行（〇/○/◯ ＋ 今後 ＋ のこと/について/ついて/:ついて/なし）。「今後の方針」等は含めない
KONGO_HEAD = re.compile(r"^\s*[〇○◯]\s*今後\s*(?:の\s*こと|について|ついて|:\s*ついて)?\s*[:;]?\s*$")
SEC_MARK   = re.compile(r"^\s*[#〇○◯]")
BLANK_LINE = re.compile(r"^[\s?]*$")   # 空行。「?」のみの行（取り込み時の文字化けと思われる）も空行扱い

# ACP 系の語：index 前 ACP_WIN 日以内に「新しい文」として現れたものを目視候補にする
ACP_WIN = 7
ACP = {
    "DNAR": r"(?<![A-Za-z])DNA?R(?![A-Za-z])",
    "延命": r"延命",
    "蘇生": r"蘇生|CPR|心マ",
    # 人工呼吸器は同じ文に意向・方針の語がある場合のみ（機器管理の記載を除く）
    "挿管・人工呼吸器": r"挿管|^(?=.*(?:希望|望ま|行わない|しない|拒否)).*?人工呼吸器",
    "搬送しない": r"(?:搬送|救急車).{0,10}(?:しない|希望(?:せず|なし|しない)|呼ばない)",
    "突然死": r"突然死",
    # 時・場合・際と組の場合のみ（心肺停止の発生報告を除く）
    "心肺停止": r"心肺停止(?:時|の(?:場合|際|時)|し(?:た)?(?:場合|際|時)|となった(?:場合|際|時)|になった(?:場合|際|時))",
    "人生会議": r"人生会議",
}
ACP_RE = [(t, re.compile(p, re.I)) for t, p in ACP.items()]
SPLIT_RE = re.compile(r"[。\r\n]+")
# 退院後の書き換えの目安（参考フラグ）：退院時会議の記録、または本文に退院の記載がある記録が DISCH_WIN 日以内にある
DISCH_WIN = 14
DISCH_RE  = re.compile(r"本日退院|退院し|退院され|退院とな|退院後初|退院後の初")

# 第1段階（ARIS）
THRESH      = {"24h": 24, "48h": 48, "72h": 72, "7d": 168}   # 判定の閾値（index の何時間以上前か）
S1_MAX_HITS = 3     # 第1段階の目視ファイルに載せる、患者ごとのヒット数（古い順）

def nfkc(x):
    return unicodedata.normalize("NFKC", x) if isinstance(x, str) else ""

## 1. 対象患者と index

In [ ]:
pop = pd.read_csv(POP_CSV, dtype=str, encoding="utf-8-sig")
pop["Patient_ID"] = pop["Patient_ID"].str.strip().str.replace(r"\.0$", "", regex=True)
assert not pop["Patient_ID"].duplicated().any()
pop["dd"]         = pd.to_datetime(pop["dd"], errors="coerce")
pop["index_date"] = pd.to_datetime(pop["index_date"], errors="coerce")
print("対象:", len(pop))
print(pop["index_type"].value_counts(dropna=False))
print("index_date 欠損:", pop["index_date"].isna().sum())

con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
con.execute("CREATE TEMP TABLE ids (Patient_ID TEXT PRIMARY KEY)")
con.executemany("INSERT INTO temp.ids VALUES (?)", [(p,) for p in pop["Patient_ID"]])

## 2. カルテ読込（全 visit_type）

In [ ]:
kt = pd.read_sql('''
    SELECT k.record_no, CAST(CAST(k.Patient_ID AS INTEGER) AS TEXT) AS Patient_ID,
           k.visit_type, k.visit_datetime, k.karte_text
    FROM karte k JOIN temp.ids i ON CAST(CAST(k.Patient_ID AS INTEGER) AS TEXT) = i.Patient_ID
''', con)
con.close()

s = (kt["visit_datetime"].astype(str)
       .str.replace(r"\([^)]*\)", " ", regex=True)
       .str.replace(r"\s+", " ", regex=True).str.strip())
kt["rec_dt"]   = pd.to_datetime(s, format="%Y/%m/%d %H:%M", errors="coerce")
kt["rec_date"] = pd.to_datetime(s.str.extract(r"^(\d{4}/\d{1,2}/\d{1,2})")[0], format="%Y/%m/%d", errors="coerce")
print("記録:", len(kt), "/ 日付パース失敗:", kt["rec_date"].isna().sum(), "/ 時刻パース失敗:", kt["rec_dt"].isna().sum())
print(kt.loc[kt["rec_date"].isna(), "visit_datetime"].head(10).tolist())

vt = kt["visit_type"].fillna("")
kt["source"] = np.select([vt.isin(DOC_VISIT_TYPES), vt.isin(NURSE_TYPES), vt.eq("文書")],
                         ["医師訪問", "看護", "文書"], default="その他")
kt["text_n"] = kt["karte_text"].map(nfkc)
print(pd.crosstab(kt["visit_type"].fillna("NULL"), kt["source"]).loc[lambda d: d.sum(axis=1).sort_values(ascending=False).index])

## 3. 死亡時刻（搬送なし）と index 日時
- 死亡日当日の医師訪問記録（DEATH_VISIT_TYPES）で DEATH_WORDS を含む最も早い記録の時刻

In [ ]:
kt = kt.merge(pop[["Patient_ID", "dd"]], on="Patient_ID", how="inner")
dw = "|".join(map(re.escape, DEATH_WORDS))
dc = kt[(kt["rec_date"] == kt["dd"]) & kt["visit_type"].isin(DEATH_VISIT_TYPES)
        & kt["text_n"].str.contains(dw) & kt["rec_dt"].notna()]
death_dt = dc.groupby("Patient_ID")["rec_dt"].min().rename("death_dt")
pop = pop.drop(columns="death_dt", errors="ignore").merge(death_dt, on="Patient_ID", how="left")

tr_dt = pd.to_datetime(pop["index_date"].dt.strftime("%Y-%m-%d") + " " + pop["index_time"].str.strip().str[:5],
                       format="%Y-%m-%d %H:%M", errors="coerce")   # index_time 欠損は NaT
print("搬送の時刻あり:", tr_dt[pop["index_type"] == "搬送"].notna().sum(), "/ index_time 記入あり:",
      pop.loc[pop["index_type"] == "搬送", "index_time"].notna().sum())
pop["index_dt"] = np.where(pop["index_type"] == "搬送", tr_dt,
                  np.where(pop["index_type"] == "死亡", pop["death_dt"], pd.NaT))
pop["index_dt"] = pd.to_datetime(pop["index_dt"])
pop["index_unit"] = np.where(pop["index_dt"].notna(), "hour", "day")
pop["index_dt_source"] = np.select(
    [(pop["index_type"] == "搬送") & pop["index_dt"].notna(),
     (pop["index_type"] == "死亡") & pop["index_dt"].notna()],
    [pop["index_time_source"], "死亡確認カルテの記録時刻"], default="なし（日単位）")
print(pd.crosstab(pop["index_type"], pop["index_unit"], margins=True))
mm = pop["main_analysis"] == "True"
print("主解析のみ\n", pd.crosstab(pop.loc[mm, "index_type"], pop.loc[mm, "index_unit"], margins=True))

## 4. 窓（index までの記録）
- hour：rec_dt < index_dt（rec_dt 欠損は rec_date < index_date、当日は `same_day=1` で含める）
- day：rec_date ≤ index_date（当日は `same_day=1`）

In [ ]:
kt = kt.merge(pop[["Patient_ID", "index_type", "index_date", "index_dt", "index_unit"]], on="Patient_ID", how="left")
hour = kt["index_unit"].eq("hour")
has_t = kt["rec_dt"].notna()
before_day = kt["rec_date"] < kt["index_date"]
same_day   = kt["rec_date"] == kt["index_date"]
in_win = np.where(hour & has_t, kt["rec_dt"] < kt["index_dt"], before_day | same_day)
kt["same_day"] = (same_day & ~(hour & has_t)).astype(int)
w = kt[in_win & kt["rec_date"].notna()].copy()
w["lead_days"]  = (w["index_date"] - w["rec_date"]).dt.days
w["lead_hours"] = ((w["index_dt"] - w["rec_dt"]).dt.total_seconds() / 3600).round(1)
print("窓内記録:", len(w), "/ 患者:", w["Patient_ID"].nunique(), "/", len(pop))
print(pd.crosstab(w["source"], w["index_unit"]))
nrec = w.groupby("Patient_ID").size()
print("窓内記録なし:", (~pop["Patient_ID"].isin(nrec.index)).sum())

## 5. ARIS（全期間）

In [ ]:
KEEP = ["Patient_ID", "record_no", "source", "visit_type", "visit_datetime", "rec_dt", "rec_date",
        "lead_days", "lead_hours", "same_day", "index_type", "index_unit"]
rows = []
for r in w.itertuples(index=False):
    ms = list(ARIS_RE.finditer(r.text_n))
    if not ms:
        continue
    t = r.text_n
    m0 = ms[0]
    rows.append({**{c: getattr(r, c) for c in KEEP},
                 "n_match": len(ms), "matches": "|".join(sorted({m.group() for m in ms})),
                 "yotei": int(any("予定" in t[max(0, m.start() - YOTEI_WIN): m.end() + YOTEI_WIN] for m in ms)),
                 "context": t[max(0, m0.start() - CTX): m0.end() + CTX].replace("\r\n", " / ").replace("\r", " / ").replace("\n", " / ")})
aris = pd.DataFrame(rows, columns=KEEP + ["n_match", "matches", "yotei", "context"])
aris = aris.sort_values(["Patient_ID", "rec_dt", "record_no"]).reset_index(drop=True)
aris["hit_order"] = aris.groupby("Patient_ID").cumcount() + 1
print("ARIS ヒット記録:", len(aris), "/ 患者:", aris["Patient_ID"].nunique())
print(aris["matches"].value_counts().head(20))
print(pd.crosstab(aris["source"], aris["yotei"]))

## 6. 暫定種語 A/B/C（index 前 SEED_WIN 日以内）

In [ ]:
rows = []
for r in w[w["lead_days"] <= SEED_WIN].itertuples(index=False):
    for cat, term, rx in SEED_RE:
        for m in rx.finditer(r.text_n):
            s_, e_ = m.start(), m.end()
            rows.append({**{c: getattr(r, c) for c in KEEP},
                         "category": cat, "term": term, "matched": m.group(0),
                         "negated": int(bool(NEG_RE.search(r.text_n[e_:e_ + 12]))),
                         "context": r.text_n[max(0, s_ - CTX): e_ + CTX].replace("\r\n", " / ").replace("\r", " / ").replace("\n", " / ")})
seed = pd.DataFrame(rows, columns=KEEP + ["category", "term", "matched", "negated", "context"])
seed = seed.sort_values(["Patient_ID", "rec_dt", "record_no", "category", "term"]).reset_index(drop=True)
print("種語ヒット:", len(seed), "/ 記録:", seed["record_no"].nunique(), "/ 患者:", seed["Patient_ID"].nunique())
print(seed.groupby(["category", "term"])["Patient_ID"].nunique().rename("患者数"))

## 7. 「今後のこと／今後について」欄の文言変化（定期訪問のみ）
- 欄の終わり：空行（「?」のみの行を含む）の次の行が # か 〇○◯ で始まる位置（なければ記録末尾）
- 比較は空白を除いた文字列。同じ患者の直前の欄（全期間）と異なれば `changed=1`
- 最初の欄は比較対象なし（changed=0, first_section=1）

In [ ]:
def kongo_section(t):
    lines = re.split(r"\r\n|\r|\n", t)
    for i, l in enumerate(lines):
        if KONGO_HEAD.match(l):
            body = []
            for j in range(i + 1, len(lines)):
                if j > i + 1 and SEC_MARK.match(lines[j]) and BLANK_LINE.match(lines[j - 1]):
                    break
                body.append(lines[j])
            return lines[i].strip(), "\n".join(body).strip()
    return None, None

kg = w[w["visit_type"] == KONGO_VISIT_TYPE].copy()
kg[["k_head", "k_text"]] = kg["text_n"].apply(lambda t: pd.Series(kongo_section(t)))
kg = kg[kg["k_head"].notna()].sort_values(["Patient_ID", "rec_dt", "record_no"]).copy()
kg["k_norm"] = kg["k_text"].str.replace(r"\s+", "", regex=True)
kg["prev_text"] = kg.groupby("Patient_ID")["k_text"].shift(1)
prev_norm = kg.groupby("Patient_ID")["k_norm"].shift(1)
kg["first_section"] = prev_norm.isna().astype(int)
kg["changed"] = (prev_norm.notna() & (kg["k_norm"] != prev_norm)).astype(int)
print("今後欄のある定期訪問記録:", len(kg), "/ 患者:", kg["Patient_ID"].nunique())
print("今後欄の見出し:\n", kg["k_head"].value_counts())
print("空の欄:", (kg["k_norm"] == "").sum())

kch = kg[kg["changed"] == 1]
if KONGO_WIN is not None:
    kch = kch[kch["lead_days"] <= KONGO_WIN]
pn = kch["prev_text"].fillna("").str.replace(r"\s+", "", regex=True)
kch["append_only"] = [int(b.startswith(a)) for a, b in zip(pn, kch["k_norm"])]   # 直前の欄に追記しただけ
kch["added_text"]  = [b[len(a):] if f else "" for a, b, f in zip(pn, kch["k_norm"], kch["append_only"])]
kch = kch[KEEP + ["k_head", "append_only", "added_text", "prev_text", "k_text"]].reset_index(drop=True)
print("うち追記のみ:", kch["append_only"].sum())
print(f"変化あり（窓内）: {len(kch)} 記録 / {kch['Patient_ID'].nunique()} 患者")

## 8. ACP 系の語の変化（index 前 ACP_WIN 日以内）
- ACP 語を含む文が、その患者のそれ以前の全記録の本文（空白除去）に部分文字列として含まれなければ「新しい文」＝候補
  - 文の削除・短縮だけの書き換えは候補にならない（短縮後の文は以前の本文に含まれるため）
- `first_term=1`：その語がこの患者で初めて出た
- `sim_prev`：同じ語を含む以前の文との最大類似度（0〜1）。高いほど言い回しの手直しに近い。`prev_similar` がその文
- `after_discharge=1`：DISCH_WIN 日以内に退院の記録あり（退院後の書き直しの目安。除外はしない）

In [ ]:
from difflib import SequenceMatcher
NS = lambda s: re.sub(r"\s+", "", s)

wa = w.sort_values(["Patient_ID", "rec_date", "rec_dt", "record_no"])
disch = wa[(wa["visit_type"] == "退院時会議") | wa["text_n"].str.contains(DISCH_RE)].groupby("Patient_ID")["rec_date"].apply(list)

acp_rows = []
for pid, g in wa.groupby("Patient_ID", sort=False):
    if not (g["lead_days"] <= ACP_WIN).any():
        continue
    prior = ""                      # それ以前の全記録の本文（空白除去）
    prev_s = {t: [] for t in ACP}   # 語ごとの以前の文
    seen = set()
    dd_list = disch.get(pid, [])
    for r in g.itertuples(index=False):
        sents = [s.strip() for s in SPLIT_RE.split(r.text_n) if s.strip()]
        for s in sents:
            ts = [t for t, rx in ACP_RE if rx.search(s)]
            if not ts:
                continue
            if r.lead_days <= ACP_WIN and NS(s) not in prior:
                cand = {x for t in ts for x in prev_s[t]}
                best, sim = "", 0.0
                for x in cand:
                    q = SequenceMatcher(None, x, s).ratio()
                    if q > sim:
                        best, sim = x, q
                acp_rows.append({**{c: getattr(r, c) for c in KEEP},
                                 "terms": "|".join(ts), "first_term": int(any(t not in seen for t in ts)),
                                 "sim_prev": round(sim, 2), "after_discharge":
                                     int(any(0 <= (r.rec_date - d).days <= DISCH_WIN for d in dd_list)),
                                 "sentence": s, "prev_similar": best})
        for s in sents:
            for t, rx in ACP_RE:
                if rx.search(s):
                    prev_s[t].append(s)
                    seen.add(t)
        prior += NS(r.text_n)

acp = pd.DataFrame(acp_rows, columns=KEEP + ["terms", "first_term", "sim_prev", "after_discharge", "sentence", "prev_similar"])
acp = acp.drop_duplicates(["Patient_ID", "record_no", "sentence"]).reset_index(drop=True)
print(f"ACP 新しい文: {len(acp)} 文 / {acp['record_no'].nunique()} 記録 / {acp['Patient_ID'].nunique()} 患者")
print("語の初出:", acp["first_term"].sum(), "/ 退院後フラグ:", acp["after_discharge"].sum())
print(acp["terms"].value_counts().head(15))
print("sim_prev 分布:\n", acp["sim_prev"].describe())

## 9. 出力

In [ ]:
def to_csv(df, name):
    df = df.copy()
    df.insert(0, "valid", "")   # 1=該当, 0=非該当
    df.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    print(name, len(df))

to_csv(aris, "1_aris_hits_review.csv")
to_csv(seed, "2_seed_hits_review.csv")
to_csv(kch,  "3_kongo_changes_review.csv")
to_csv(acp,  "4_acp_changes_review.csv")

def first_hit(df, label):
    if df.empty:
        return pd.DataFrame(columns=["Patient_ID"])
    f = df.sort_values(["Patient_ID", "rec_dt", "record_no"]).groupby("Patient_ID").head(1)
    out = f[["Patient_ID", "rec_dt", "rec_date", "lead_days", "lead_hours", "source"]].copy()
    out.columns = ["Patient_ID"] + [f"{label}_{c}" for c in out.columns[1:]]
    out[f"{label}_n_records"] = out["Patient_ID"].map(df.groupby("Patient_ID")["record_no"].nunique())
    return out

summ = pop[["Patient_ID", "dd", "death_place", "index_type", "index_date", "index_dt", "index_unit",
            "index_dt_source", "death_dt", "main_analysis", "exclude_cutoff30",
            "exclude_transport_unknown", "exclude_care_ended"]].copy()
summ["n_records_in_window"] = summ["Patient_ID"].map(nrec).fillna(0).astype(int)
for df, lab in [(aris, "aris"), (aris[aris["yotei"] == 0], "aris_noyotei"), (seed, "seed"), (kch, "kongo"), (acp, "acp")]:
    summ = summ.merge(first_hit(df, lab), on="Patient_ID", how="left")
summ.to_csv(OUT / "5_patient_summary_unverified.csv", index=False, encoding="utf-8-sig")

def count(d):
    c = ["aris_rec_date", "seed_rec_date", "kongo_rec_date", "acp_rec_date"]
    return [len(d)] + [d[x].notna().sum() for x in c] + [d[c].notna().any(axis=1).sum()]
m = summ["main_analysis"] == "True"
tab = pd.DataFrame({"全例": count(summ), "主解析": count(summ[m])},
                   index=["患者数", "ARISヒットあり", "種語ヒットあり", "今後欄変化あり", "ACP変化あり", "いずれかあり"])
print(tab)
print("\n最初のARISヒット（未検証）の index 前日数：主解析")
print(summ.loc[m, "aris_lead_days"].describe())
print("\n出力先:", OUT)

---
## 10. 第1段階：ARIS の目視ファイル
- `stage1_aris_review.csv`：患者ごとに古い順に最大 S1_MAX_HITS 件
- 目視：各患者の `hit_order`=1 の `valid` に 1（該当）か 0（非該当）を記入。0 のときだけ次の行を見る。最初の 1 より後は空欄でよい
- 「予定」は該当（1）に含める。`yotei`=1 で区別できる
- `type_if_valid`：その行を最初の有効なARISとした場合の型
  - A：index の7日以上前／B1：72時間以上前〜7日前／B2：48時間以上前〜72時間前／B3：24時間以上前〜48時間前／C：index 前24時間以内
- 日単位（`lead_unit`=day）は「index 日−記録日」の日数で判定（1日＝24時間、2日＝48時間、3日＝72時間、7日）
- 記入済みのファイルがあれば上書きしない（`stage1_aris_review_new.csv` に出す）

In [ ]:
def a_type(h, d, unit):
    ok = {k: (h >= v) if unit == "hour" else (d >= v // 24) for k, v in THRESH.items()}
    return "A" if ok["7d"] else "B1" if ok["72h"] else "B2" if ok["48h"] else "B3" if ok["24h"] else "C"

s1 = aris.copy()
s1["n_hits_total"] = s1.groupby("Patient_ID")["hit_order"].transform("max")
s1 = s1[s1["hit_order"] <= S1_MAX_HITS].copy()
s1["lead_unit"] = np.where(s1["lead_hours"].notna(), "hour", "day")
s1["type_if_valid"] = [a_type(h, d, u) for h, d, u in zip(s1["lead_hours"], s1["lead_days"], s1["lead_unit"])]
s1 = s1.merge(pop[["Patient_ID", "main_analysis"]], on="Patient_ID", how="left")
s1.insert(0, "valid", "")
s1 = s1[["valid", "Patient_ID", "main_analysis", "hit_order", "n_hits_total", "rec_dt", "source", "visit_type",
         "yotei", "matches", "lead_days", "lead_hours", "lead_unit", "type_if_valid", "context",
         "record_no", "index_type", "index_unit", "same_day"]]

S1_FILE = OUT / "stage1_aris_review.csv"
if S1_FILE.exists() and pd.read_csv(S1_FILE, dtype=str, encoding="utf-8-sig")["valid"].notna().any():
    S1_FILE = OUT / "stage1_aris_review_new.csv"
    print("記入済みの stage1_aris_review.csv があるため上書きしません →", S1_FILE.name)
s1.to_csv(S1_FILE, index=False, encoding="utf-8-sig")
print(S1_FILE.name, len(s1), "行 /", s1["Patient_ID"].nunique(), "患者（hit_order=1 は",
      (s1["hit_order"] == 1).sum(), "行）")
print("hit_order=1 を有効とした場合の型（主解析）:\n",
      s1[(s1["hit_order"] == 1) & (s1["main_analysis"] == "True")]["type_if_valid"].value_counts().sort_index())

---
## 11. 第1段階の確定：`stage1_aris_review.csv` の目視後に実行
- 別の日に実行する場合は `REVIEW` のパスを目視したファイルに書き換える（以降のセルは 0〜3 を先に実行しておく）
- 出力 `stage1_result.csv`（1患者1行）
  - `aris_type`：A／B1／B2／B3／C／D（有効なARISなし）／未確認（記入漏れ）
  - `aris_first_dt`・`aris_first_lead_*`：最初の有効なARIS
  - `aris_first_yotei`：最初の有効なARISが「予定」の記載
  - `aris_border`：日単位で閾値ちょうどの例（境界フラグ）
  - `stage2_needed`・`stage2_thresh`：第2段階で探す範囲（index の何時間以上前の記録か）
  - `stage2_end_dt`（時刻単位）／`stage2_end_date`（日単位、この日以前の記録）

In [ ]:
REVIEW = OUT / "stage1_aris_review.csv"   # 目視済みファイル
STAGE2_THRESH = {"A": None, "B1": 168, "B2": 72, "B3": 48, "C": 24, "D": 24}

rv = pd.read_csv(REVIEW, dtype=str, encoding="utf-8-sig")
rv["valid"] = pd.to_numeric(rv["valid"], errors="coerce")
rv["hit_order"] = rv["hit_order"].astype(int)
rv["n_hits_total"] = rv["n_hits_total"].astype(int)
bad = rv[rv["valid"].notna() & ~rv["valid"].isin([0, 1])]
assert bad.empty, f"valid は 0/1 のみ:\n{bad[['Patient_ID', 'hit_order', 'valid']]}"

# 最初の有効なARIS
fv = (rv[rv["valid"] == 1].sort_values(["Patient_ID", "hit_order"]).groupby("Patient_ID").head(1)
        .set_index("Patient_ID"))
# 記入漏れ：最初の 1 より前に空欄がある／1 がなく空欄が残る
first_order = fv["hit_order"]
rv["first_order"] = rv["Patient_ID"].map(first_order)
gap = rv[rv["valid"].isna() & (rv["first_order"].isna() | (rv["hit_order"] < rv["first_order"]))]
unres = set(gap["Patient_ID"])
# ファイル内の全ヒットが 0 で、ファイル外にまだヒットがある
all0 = rv.groupby("Patient_ID").agg(n0=("valid", lambda s: (s == 0).sum()), n=("valid", "size"),
                                     tot=("n_hits_total", "first"))
more = set(all0[(all0["n0"] == all0["n"]) & (all0["tot"] > all0["n"])].index)
print("記入漏れの患者:", len(unres), "/ 全0でファイル外にヒットが残る患者:", len(more))
if unres or more:
    pd.DataFrame({"Patient_ID": sorted(unres | more)}).assign(
        reason=lambda d: np.where(d["Patient_ID"].isin(unres), "記入漏れ", "全0・ファイル外にヒットあり（1_aris_hits_review.csv を確認）")
    ).to_csv(OUT / "stage1_unresolved.csv", index=False, encoding="utf-8-sig")

res = pop[["Patient_ID", "main_analysis", "index_type", "index_date", "index_dt", "index_unit"]].copy()
res["aris_type"] = res["Patient_ID"].map(fv["type_if_valid"])
res.loc[res["aris_type"].isna(), "aris_type"] = "D"
res.loc[res["Patient_ID"].isin(unres | more) & ~res["Patient_ID"].isin(fv.index), "aris_type"] = "未確認"
res["aris_first_dt"]         = res["Patient_ID"].map(fv["rec_dt"])
res["aris_first_lead_hours"] = res["Patient_ID"].map(fv["lead_hours"])
res["aris_first_lead_days"]  = res["Patient_ID"].map(fv["lead_days"])
res["aris_first_unit"]       = res["Patient_ID"].map(fv["lead_unit"])
res["aris_first_yotei"]      = res["Patient_ID"].map(fv["yotei"])
res["aris_first_source"]     = res["Patient_ID"].map(fv["source"])
ld = pd.to_numeric(res["aris_first_lead_days"], errors="coerce")
res["aris_border"] = [",".join(k for k, v in THRESH.items() if u == "day" and d == v // 24) if u == "day" else ""
                      for u, d in zip(res["aris_first_unit"], ld)]

res["stage2_thresh"] = res["aris_type"].map(STAGE2_THRESH)
res["stage2_needed"] = res["stage2_thresh"].notna().astype(int)
res.loc[res["aris_type"] == "未確認", "stage2_needed"] = np.nan
th = pd.to_numeric(res["stage2_thresh"], errors="coerce")
hour = res["index_unit"] == "hour"
res["stage2_end_dt"]   = np.where(hour & th.notna(), res["index_dt"] - pd.to_timedelta(th, unit="h"), pd.NaT)
res["stage2_end_date"] = np.where(~hour & th.notna(), res["index_date"] - pd.to_timedelta(th // 24, unit="D"), pd.NaT)
res["stage2_end_dt"]   = pd.to_datetime(res["stage2_end_dt"])
res["stage2_end_date"] = pd.to_datetime(res["stage2_end_date"]).dt.date
res.to_csv(OUT / "stage1_result.csv", index=False, encoding="utf-8-sig")

order = ["A", "B1", "B2", "B3", "C", "D", "未確認"]
m = res["main_analysis"] == "True"
tab = pd.DataFrame({"全例": res["aris_type"].value_counts(), "主解析": res.loc[m, "aris_type"].value_counts(),
                    "主解析_うち予定": res.loc[m & (res["aris_first_yotei"] == "1"), "aris_type"].value_counts()}
                   ).reindex(order).fillna(0).astype(int)
print(tab)
print("\n第2段階の対象（主解析）:", int(res.loc[m, "stage2_needed"].sum()))
print("境界フラグ（主解析）:", (res.loc[m, "aris_border"] != "").sum())

---
## 12. 第2段階a：今後欄の変化・ACP の変化（stage1_result.csv の stage2_needed=1 の例）
- 探す範囲：`stage2_end` より前の記録（時刻単位は記録時刻 ≤ stage2_end_dt。記録時刻がなければ前日まで。日単位は記録日 ≤ stage2_end_date）
- 今後欄：index 前 `S2_KONGO_WIN` 日以内の変化（None で全期間）。ACP：index 前 ACP_WIN 日以内の新しい文
- `stage2a_review.csv`：患者ごとに古い順。各患者の古い行から見て、最初に該当したら `valid`=1、それ以降は空欄でよい。非該当は 0
  - `kind`：今後欄／ACP
  - `new_text`：今後欄は追記部分（追記のみの場合）または変化後の欄全体、ACP は新しい文
  - `prev_text`：今後欄は直前の欄、ACP は最も似た以前の文

In [ ]:
S2_KONGO_WIN = KONGO_WIN   # None で全期間
res1 = pd.read_csv(OUT / "stage1_result.csv", dtype=str, encoding="utf-8-sig")
tgt = res1[pd.to_numeric(res1["stage2_needed"], errors="coerce") == 1][["Patient_ID", "aris_type", "stage2_thresh", "stage2_end_dt", "stage2_end_date"]].copy()
tgt["stage2_end_dt"]   = pd.to_datetime(tgt["stage2_end_dt"], errors="coerce")
tgt["stage2_end_date"] = pd.to_datetime(tgt["stage2_end_date"], errors="coerce")
print("第2段階の対象:", len(tgt)); print(tgt["aris_type"].value_counts().sort_index())

def in_range(df):
    x = df.merge(tgt, on="Patient_ID", how="inner")
    hour = x["stage2_end_dt"].notna()
    ok = np.where(hour,
                  np.where(x["rec_dt"].notna(), x["rec_dt"] <= x["stage2_end_dt"], x["rec_date"] < x["stage2_end_dt"].dt.normalize()),
                  x["rec_date"] <= x["stage2_end_date"])
    return x[ok].copy()

# 今後欄：全期間の変化から S2_KONGO_WIN で絞る
kc = kg[kg["changed"] == 1].copy()
if S2_KONGO_WIN is not None:
    kc = kc[kc["lead_days"] <= S2_KONGO_WIN]
pn = kc["prev_text"].fillna("").str.replace(r"\s+", "", regex=True)
kc["append_only"] = [int(b.startswith(a)) for a, b in zip(pn, kc["k_norm"])]
kc["new_text"] = [b[len(a):] if f else t for a, b, f, t in zip(pn, kc["k_norm"], kc["append_only"], kc["k_text"])]
kc = in_range(kc)
kc["kind"] = "今後欄"

ac = in_range(acp.copy())
ac["kind"] = "ACP"
ac = ac.rename(columns={"sentence": "new_text", "prev_similar": "prev_text"})

S2COLS = ["Patient_ID", "aris_type", "stage2_thresh", "kind", "rec_dt", "rec_date", "lead_days", "lead_hours",
          "source", "visit_type", "new_text", "prev_text", "append_only", "terms", "first_term", "sim_prev",
          "after_discharge", "record_no", "same_day", "stage2_end_dt", "stage2_end_date"]
s2 = pd.concat([kc, ac], ignore_index=True).reindex(columns=S2COLS)
s2 = s2.sort_values(["Patient_ID", "rec_date", "rec_dt", "kind"]).reset_index(drop=True)
s2.insert(0, "order", s2.groupby("Patient_ID").cumcount() + 1)
s2.insert(0, "valid", "")
for c in ["new_text", "prev_text"]:
    s2[c] = s2[c].fillna("").str.replace(r"\r\n|\r|\n", " / ", regex=True)

S2_FILE = OUT / "stage2a_review.csv"
if S2_FILE.exists() and pd.read_csv(S2_FILE, dtype=str, encoding="utf-8-sig")["valid"].notna().any():
    S2_FILE = OUT / "stage2a_review_new.csv"
    print("記入済みの stage2a_review.csv があるため上書きしません →", S2_FILE.name)
s2.to_csv(S2_FILE, index=False, encoding="utf-8-sig")

mm = s2["Patient_ID"].isin(res1.loc[res1["main_analysis"] == "True", "Patient_ID"])
print(S2_FILE.name, len(s2), "行 /", s2["Patient_ID"].nunique(), "患者（主解析:", mm.sum(), "行 /", s2.loc[mm, "Patient_ID"].nunique(), "患者）")
print(pd.crosstab(s2["aris_type"], s2["kind"], margins=True))
print("候補なしの対象患者:", len(set(tgt["Patient_ID"]) - set(s2["Patient_ID"])))